# Semaine 9 — Jour 7 : Projet RAG : preuves, abstention et évaluation

Assembler ingestion, recherche hybride, reranking et contexte cité.

Parcours essentiel : 60 minutes (cours 20 min, lab 25 min, exercice 10 min, synthèse 5 min). Prévoir 2 à 3 heures pour l’unité complète avec entretien et challenge. Les modèles et services optionnels ajoutent un temps d’installation distinct.

Prérequis : Python 3.11+, JSON, notions de vecteurs et de persistance. Le parcours obligatoire utilise uniquement la bibliothèque standard et ne contacte aucun service externe.

Lire [les objectifs](learning_objectives.md), [le chapitre](chapter.md), puis [le lab](labs/README.md). Réaliser [les exercices](exercises.md), [l’entretien](interview.md) et [le challenge](challenge.md) avant les corrigés locaux.

Notebooks depuis la racine : `notebooks/week09/S9_J7_projet_rag.ipynb` et `notebooks/week09/teacher/S9_J7_projet_rag_teacher.ipynb`.

Critère de réussite : Évaluer retrieval et réponse avec des métriques et limites distinctes.

Les calculs de retrieval et de persistance sont exécutables; le corpus est fictif. L’encodeur de base est lexical et la réponse finale est extractive. Les extensions neuronales sont séparées et n’ont pas été exécutées ici.


# Objectifs mesurables

- Assembler ingestion, recherche hybride, reranking et contexte cité.
- Vérifier les références, l’isolation et l’abstention quand les preuves manquent.
- Évaluer retrieval et réponse avec des métriques et limites distinctes.

## Preuves attendues

Une sortie reproductible, un contrôle d’invariant et une limite du mécanisme correctement expliquée.


# S9 J7 — Projet RAG : preuves, abstention et évaluation


## 1. Un pipeline complet de connaissances
Le projet reçoit une question, sélectionne les passages autorisés, fusionne les classements, reclassifie les candidats, applique un seuil explicite et assemble un contexte borné. La réponse locale reproduit les extraits avec leurs références. Le projet démontre le pipeline de retrieval et le contrat d’un RAG; il ne contacte pas de modèle génératif.

Cette distinction est volontaire : l’encodage hashbow, BM25, RRF, SQLite, les filtres et les métriques s’exécutent réellement. La synthèse par LLM reste un raccord à construire après évaluation du retrieval. Le port neuronal optionnel ne transforme pas la réponse extractive en une génération.

## 2. Une citation vérifiable
Chaque référence contient chunk_id, doc_id, version, start, end, text et source. citation_valid charge le chunk courant dans le tenant et compare les champs. Une citation inventée, modifiée ou liée à une ancienne version échoue. La réponse par défaut ne produit que des extraits exacts, ce qui rend le contrat facile à inspecter.

Une référence valide ne prouve pas que le document est juste. Avec un LLM, elle ne prouverait pas non plus que toutes ses affirmations sont soutenues par le passage cité. Il faudrait évaluer fidélité, couverture des affirmations, contradictions et utilité. Ne pas nommer « zéro hallucination » un simple test d’identifiants de sources.

## 3. Abstention et limites lexicales
Le moteur peut s’abstenir si aucun passage ne satisfait le seuil de couverture ou si le budget ne permet pas d’en conserver. Deux questions hors corpus servent de cas d’abstention. Une absence de réponse peut aussi être un faux négatif : q8 paraphrase une question connue mais utilise des mots éloignés de la baseline.

Un système réel doit distinguer absence d’information, retrieval raté, autorisation insuffisante et budget trop faible. La réponse locale regroupe ces situations sous abstain et explique sa portée. Améliorer ce diagnostic constitue une extension, pas une garantie cachée. Aucun document team-b n’est ajouté pour remplir un contexte team-a trop pauvre.

## 4. Mesures de retrieval
Le jeu contient huit requêtes positives avec qrels au niveau document. Recall@k est la fraction des documents pertinents retrouvés dans les k premiers documents distincts. MRR@k est l’inverse du rang du premier pertinent dans cette coupure, sinon zéro. nDCG@k compare le gain actualisé observé au meilleur classement possible; les jugements sont binaires dans ce lab.

Les requêtes sans document pertinent ne sont pas forcées dans une formule au dénominateur nul. Elles alimentent le contrôle d’abstention. Le rapport conserve les résultats par requête, les moyennes, l’empreinte du corpus, des qrels et de l’encodeur. Les questions constituent un jeu de développement minuscule, sans échantillon de validation réservé.

## 5. Comparer les variantes sans inventer un gain
Les modes lexical, vector, hybrid et reranked sont évalués sur les mêmes questions. Un résultat identique ou une dégradation doit rester dans le rapport. Le cas q8 rend visible que les canaux lexicaux ne démontrent pas une capacité sémantique. Ajouter un modèle appris réclame une nouvelle mesure, pas une réutilisation des scores de hashbow.

La semaine 8 fournissait des principes de décision et de traçabilité. Ce projet les applique à la récupération de connaissances avec des métriques spécifiques, sans réécrire les modules partagés. Une intégration réelle peut transmettre les rapports à votre dispositif de supervision après alignement explicite des contrats.

## 6. Revue d’architecture
Présenter ingestion atomique, index exact, autorisations préalables, compatibilité d’encodeur, mémoire séparée et citations. Expliquer ensuite les limites : tenant seul, un processus SQLite, pas d’ANN, pas de génération, corpus fictif et pas de campagne de pertinence neuronale.

Pour aller plus loin, choisir une amélioration dont le besoin est mesuré : encodeur sémantique pour les paraphrases, pgvector pour une expérimentation de stockage, reranker appris pour les candidats ambigus ou générateur évalué pour la synthèse. Chaque ajout doit conserver suppression, isolation et provenance.

## À retenir
Un RAG exploitable relie recherche, autorisation, références et mesure de qualité. Une bonne démonstration expose aussi les questions qu’elle ne sait pas traiter.


## Diagramme du jour

```mermaid
flowchart TD
 A["Question et tenant autorisé"] --> B["Retrieval hybride"]
 B --> C["Reranking et budget"]
 C --> D{"Passages étayés ?"}
 D -->|non| E["Abstention"]
 D -->|oui| F["Extraits et citations"]
 F --> G["Contrôle des références"]
 H["Qrels et cas négatifs"] --> I["Évaluation par mode"]
 B --> I
```

Le rendu Mermaid dépend du lecteur; la source .mmd reste disponible.

# Lab exécutable

Les cellules s’exécutent hors ligne depuis l’arbre du dépôt.

In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
root = next((p for p in (cwd, *cwd.parents) if (p / "examples/week09/knowledge.py").is_file()), None)
assert root is not None, "Ouvrir le notebook depuis le dépôt décompressé"
sys.path.insert(0, str(root))
from examples.week09.demo import populated, evaluate_retrieval
from examples.week09.knowledge import answer
store = populated()
try:
    response = answer(store, "Comment rejouer un paiement avec Kafka ?", "team-a")
    assert response["status"] == "supported_extract" and response["citations"]
    assert all(store.citation_valid(c, "team-a") for c in response["citations"])
    assert answer(store, "Quelle météo demain à Oslo ?", "team-a")["status"] == "abstain"
    report = evaluate_retrieval(store)
    assert report["queries"] == 8
    print(response["answer"])
    print(report["summaries"])
finally:
    store.close()

# Lab J7 — Projet RAG : preuves, abstention et évaluation

Depuis la racine du dépôt :

```bash
python3 book/week09/day07/labs/run.py
python3 -m examples.week09.demo --day 7
```

Résultat attendu : **réponse extractive citée et rapport de huit requêtes sur quatre modes**.

## Manipulations

1. Reproduire la sortie et retrouver le calcul dans `examples/week09/knowledge.py`.
2. Réaliser l’exercice 2 avec une copie des données, sans modifier le corpus distribué.
3. Vérifier l’invariant étudié et expliquer une limite.

Le corpus se trouve dans `examples/week09/corpus`. Les tests automatisés sont sous `tests/week09`. [QUICKSTART](../../../../docs/week09/QUICKSTART.md) décrit les commandes et les codes retour.

Les modèles locaux et pgvector sont optionnels : lire `examples/week09/optional/README.md` depuis la racine du dépôt. Ils ne sont pas nécessaires pour finir cette journée et ne sont pas présentés comme validés dans cet environnement.


# Exercices

Conserver votre première réponse avant de lire les solutions.

## Exercice 1 — Calculer les métriques

Pertinents={A,C}, classement=[B,C,A], k=2 : calculer Recall, MRR et nDCG binaires.

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 2 — Distinguer citation et vérité

Que prouve une citation dont le passage et les offsets sont valides ?

Livrable : réponse argumentée avec calcul, test ou observation.

## Exercice 3 — Préparer le raccord LLM

Quels contrôles faut-il ajouter avant d’utiliser une réponse synthétisée par un modèle ?

Livrable : réponse argumentée avec calcul, test ou observation.


# Entretien

Répondre en deux minutes avec un exemple concret et une limite.

## Question 1

Pourquoi le retrieval seul ne suffit-il pas à évaluer une réponse ?

## Question 2

Quelle prochaine amélioration choisir ?


# Challenge

Produire une réponse citée à la question Kafka, une abstention hors corpus et le rapport des quatre modes. Modifier une citation et vérifier le rejet. Expliquer le cas q8 plutôt que le retirer du rapport.

## Critères de réussite

Reproductibilité, provenance des données, invariant vérifié et limite expliquée sans exagérer les garanties.


# Références — J7

Sources primaires consultées le 20 octobre 2026. Les algorithmes et contrats du lab sont explicités dans le chapitre; le corpus est une création fictive du cours.

- [pgvector : documentation officielle](https://github.com/pgvector/pgvector) — stockage, distances, recherche exacte/ANN et filtres.
- [Sentence Transformers : retrieve and rerank](https://www.sbert.net/examples/sentence_transformer/applications/retrieve_rerank/README.html) — architecture de récupération en deux étapes.
- [SentenceTransformer : API](https://www.sbert.net/docs/package_reference/sentence_transformer/model.html) — encode_query, encode_document et modèles locaux.
- [CrossEncoder : API](https://www.sbert.net/docs/package_reference/cross_encoder/model.html) — scoring conjoint de paires.
- [Cormack, Clarke et Buettcher, SIGIR 2009 : Reciprocal Rank Fusion](https://research.google/pubs/reciprocal-rank-fusion-outperforms-condorcet-and-individual-rank-learning-methods/) — origine de la méthode de fusion des rangs.
- [Python sqlite3](https://docs.python.org/3/library/sqlite3.html) — persistance locale et transactions.

Lire en priorité les références du thème du jour. Les extensions évoluent; leur environnement réel doit être verrouillé et testé avant utilisation. Le paquet n’affirme pas avoir exécuté de modèle neuronal ou de serveur pgvector.
